# Hausse de loyer 2026 - Collection Équinoxe

**Question :** de combien les loyers de Collection Équinoxe augmenteront-ils en 2026?

| | |
|---|---|
| **Définition retenue** | croissance annualisée du **loyer effectif** (`sRentEffective`) **à unité constante** : médiane, sur tous les baux qui débutent en 2026 (renouvellements et relocations, six immeubles), de la hausse de chaque bail par rapport au bail précédent du même appartement |
| **Estimation 2026** | **5,3 %** (fourchette 4,3 % à 6,3 %; 3,9 % sans le facteur d'inoccupation) |
| **Méthode** | régression linéaire entraînée sur les paires de baux 2022-2025 : caractéristiques du logement et du bail précédent, plus trois facteurs publics décalés d'un an (IPC, inoccupation, chômage). Elle prédit la hausse de chaque bail de 2026, puis on en prend la médiane |
| **Validation** | backtest 2023-2025 : erreur absolue moyenne de 1,0 point sur la médiane annuelle, contre 1,4 pour la meilleure extrapolation de tendance |

### Plan, aligné sur les critères d'évaluation

| Section | Critère des consignes |
|---|---|
| 1. Définir la hausse | Définition de la hausse |
| 2. Analyser les données | Analyse des données et effet de mix |
| 3. Mesurer à unité constante | Appariement à unité constante |
| 4. Concessions | Concessions |
| 5. Renouvellements, relocations et Ontario | Renouvellements vs relocations (+ bonus Ontario) |
| 6. Données publiques | Données externes |
| 7. Prévoir et valider | Prévision et backtest (+ bonus : scénarios, par immeuble, par chambres, tableau de bord) |
| 8. Hypothèses et limites, conclusion | |
| 9. Références | |

## 0. Configuration et chargement

Toutes les constantes sont déclarées ici, une seule fois, avec leur justification : aucun nombre n'apparaît « en dur » plus bas.
Les fichiers sources ne sont jamais modifiés; toute transformation se fait en mémoire.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))   # évite un avertissement de joblib sous Windows
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.insert(0, "public_data")
from extract_cmhc import MARKETS as CMHC_MARKET_LABELS, extract_public_data, number, read_workbook

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

# --- Fichiers (lecture seule) ---
PRIVATE_DIR = Path("private_data")
RAW_PUBLIC_DIR = Path("raw_public_data")
LISTINGS_PATH = PRIVATE_DIR / "equinoxe_listings.csv"
LEASES_PATH = PRIVATE_DIR / "equinoxe_lease_history.csv"
CONCESSIONS_PATH = PRIVATE_DIR / "equinoxe_concessions.csv"
ASKING_PATH = PRIVATE_DIR / "equinoxe_asking_history.csv"
REGULATORY_RATES_PATH = Path("public_data") / "regulatory_rates.csv"
CPI_PATH = RAW_PUBLIC_DIR / "IPC.csv"
LABOUR_PATH = RAW_PUBLIC_DIR / "msrche-du-travail.csv"
MODEL_PATH = Path("models") / "modele_regression_lineaire.joblib"

# --- Clé d'unité ---
UNIT_KEY = ["sPropCode", "sUnitCode"]       # 1 pour 1 avec hUnit et sLink (section 2a)
WRONG_UNIT_KEY = ["sSite", "sUnitCode"]     # clé interdite, utilisée une seule fois pour montrer l'erreur (3b)

# --- Appariement ---
MIN_GAP_YEARS = 0.5        # paires plus rapprochées : issues d'un bail de 6 mois, non comparables à un bail annuel (3a)
MAX_GAP_YEARS = 2.5        # paires plus espacées : l'unité a pu être rénovée ou rester vide longtemps
GAP_BINS_YEARS = [0, MIN_GAP_YEARS, 0.95, 1.05, 1.6, MAX_GAP_YEARS, np.inf]   # tranches d'écart pour décrire les paires
DAYS_PER_YEAR = 365.25
PCT = 100

# --- Codes et libellés ---
RENEWAL, TURNOVER = 1, 0                  # sRenewal : 1 = locataire en place, 0 = relocation
QUEBEC, ONTARIO = "Quebec", "Ontario"     # valeurs de sState
PROMO_CHARGE_CODE = "PromoPay"
TWO_BEDROOMS = 2

# --- Périodes ---
FIRST_ANALYSIS_YEAR = 2019        # avant 2019, trop peu de paires de renouvellement
LAST_OBSERVED_YEAR = 2025         # les données s'arrêtent au 2025-12-31
FORECAST_YEAR = LAST_OBSERVED_YEAR + 1
REQUIRED_BACKTEST_YEARS = [2023, 2024, 2025]                      # exigées par les consignes
COMPARISON_YEARS = [2021, 2022] + REQUIRED_BACKTEST_YEARS         # années couvertes par les données publiques, pour les comparaisons
FIRST_PUBLIC_YEAR = 2020          # SCHL : le fichier 2021 donne aussi la variation 2020; les autres séries commencent en 2021

# --- Tolérances et paramètres d'analyse ---
RENT_MATCH_TOLERANCE = 1.0        # $ / mois : deux loyers concordent à un arrondi au dollar près
SIX_MONTH_GAP_YEARS = 0.4         # seuil alternatif qui inclut les baux de 6 mois (sensibilité, 3d)
TRIM_SHARE = 0.05                 # moyenne tronquée : retire 5 % des paires de chaque côté (sensibilité, 3d)
ONTARIO_EXEMPTION_DATE = pd.Timestamp("2018-11-15")   # première occupation après cette date : exempté de la ligne directrice

# --- Modèle (section 7) ---
PRIVATE_FEATURES = ["prev_effective_rent", "gap_years", "bedrooms"]   # connus avant le nouveau bail
PUBLIC_FACTORS = ["general_cpi", "vacancy_rate", "unemployment_rate"]  # décalés d'un an (valeur de t - 1)
CATEGORICAL_FEATURES = ["market", "lease_type"]
LEASE_TYPE_LABELS = {TURNOVER: "Relocation", RENEWAL: "Renouvellement"}   # la première catégorie sert de référence
PUBLIC_LAG_YEARS = 1              # une série publique de l'année t n'est connue qu'après t
MODEL_SELECTION_YEAR = LAST_OBSERVED_YEAR   # comparaison des familles de modèles : entraînement avant 2025, test sur 2025
RANDOM_STATE = 42
RANDOM_FOREST_PARAMS = {"n_estimators": 300, "max_depth": 8, "min_samples_leaf": 20, "random_state": RANDOM_STATE, "n_jobs": 1}
GRADIENT_BOOSTING_PARAMS = {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 20,
                            "random_state": RANDOM_STATE}

### Chargement

On charge les quatre tables et on convertit les dates une seule fois. `lease_start_year` (année de `sLeaseFrom`) est l'année
à laquelle on rattache un bail : c'est la date où le nouveau loyer **entre en vigueur**, donc celle qui compte pour les revenus.

In [ ]:
listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH, parse_dates=["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"])
concessions = pd.read_csv(CONCESSIONS_PATH, parse_dates=["sDateFrom", "sDateTo"])
asking = pd.read_csv(ASKING_PATH)

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year
asking["ask_month"] = pd.PeriodIndex(asking["sMonth"], freq="M").to_timestamp()

for name, table in [("listings", listings), ("leases", leases), ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(table):>6,} lignes  {table.shape[1]:>2} colonnes")
print(f"\nbaux du {leases['sLeaseFrom'].min():%Y-%m-%d} au {leases['sLeaseFrom'].max():%Y-%m-%d}")

## 1. Définir la hausse mesurée

« La hausse de loyer 2026 » n'est pas un seul chiffre. Les candidats (chiffrés en section 3c) :

| Définition candidate | Ce qu'elle mesure |
|---|---|
| Médiane annuelle de tous les baux | le loyer du bail « typique » de l'année : **mesure surtout le mix d'immeubles** (section 2f) |
| Même unité, loyer contractuel (`sRent`) | la hausse du prix inscrit au bail, pour un même appartement |
| **Même unité, loyer effectif (`sRentEffective`)** | **la hausse de l'argent réellement perçu, pour un même appartement** |
| Renouvellements seulement | la hausse consentie aux locataires en place (TAL au Québec, ligne directrice en Ontario) |
| Relocations seulement | la hausse du prix de marché quand une unité change de locataire |

> **Définition retenue : hausse 2026 = croissance annualisée du loyer effectif à unité constante, médiane de tous les baux
> qui débutent en 2026 (renouvellements et relocations confondus), pour les six immeubles.**

**Pourquoi :**

1. **À unité constante.** La médiane annuelle bondit de 10,8 % en 2023 alors qu'un même appartement augmente d'environ 2 % :
   elle mesure l'arrivée de trois immeubles haut de gamme, pas une hausse payée par les locataires (2f, 3c).
2. **Loyer effectif.** La question d'affaires est de **budgéter les revenus**, c'est-à-dire ce qui est perçu après les mois
   gratuits et autres concessions. Depuis 2024, les deux mesures divergent : en 2025, +7,0 % au bail mais +3,6 % perçu (section 4).
   Citer le contractuel surestimerait la hausse des revenus de plus de trois points.
3. **Tous les baux, mais modélisés selon leur type.** Les revenus 2026 viendront des renouvellements et des relocations, donc le
   chiffre les combine. Comme ils obéissent à des règles différentes (section 5), le modèle connaît le type de bail et
   pondère les deux cas.
4. **Médiane, annualisée, rattachée à `sLeaseFrom`.** La médiane résiste aux paires atypiques, l'annualisation rend comparables
   des baux de durées différentes, et `sLeaseFrom` est la date où le nouveau loyer s'applique.

**Indicateurs secondaires** rapportés à côté : renouvellements en loyer contractuel (ce que le TAL et l'Ontario encadrent),
relocations (référence pour les loyers affichés), prévision par immeuble et par nombre de chambres.

## 2. Analyser les données

Les descriptions ne correspondent pas toujours aux données. Chaque sous-section teste une hypothèse et se termine par une
**lecture** et une **conséquence**. On garde les noms Yardi pendant les vérifications, puis on renomme en 2e.

### 2a. La clé d'unité

In [ ]:
unit_key_checks = pd.Series({
    "hUnit distincts par sPropCode + sUnitCode (max)": leases.groupby(UNIT_KEY)["hUnit"].nunique().max(),
    "clés sPropCode + sUnitCode par hUnit (max)": leases.groupby("hUnit")[UNIT_KEY].nunique().max().max(),
    "clés sPropCode + sUnitCode par sLink (max)": leases.groupby("sLink")[UNIT_KEY].nunique().max().max(),
    "unités en collision sur sSite + sUnitCode": listings.duplicated(WRONG_UNIT_KEY, keep=False).sum(),
    "unités de listings": len(listings),
    "baux en double (même unité, même début)": leases.duplicated(UNIT_KEY + ["sLeaseFrom"]).sum(),
})
unit_key_checks.to_frame("valeur")

**Lecture :** `sPropCode + sUnitCode`, `hUnit` et `sLink` sont trois clés équivalentes (1 pour 1). `sSite + sUnitCode`
fusionnerait 248 appartements distincts : un site regroupe plusieurs propriétés dont les numéros d'unité se chevauchent.
Aucun doublon.

**Conséquence :** tout appariement se fait sur `UNIT_KEY = ["sPropCode", "sUnitCode"]`.

### 2b. Ce que représentent les colonnes des baux

In [ ]:
first_unit_leases = leases[leases["sTermSeq"] == 1]
lease_column_checks = pd.Series({
    "sAvailable == sLeaseFrom (% des baux)": (leases["sAvailable"] == leases["sLeaseFrom"]).mean() * PCT,
    "baux avec sTermSeq = 1": len(first_unit_leases),
    "unités distinctes": leases.groupby(UNIT_KEY).ngroups,
    "sTermSeq = 1 : % de relocations": first_unit_leases["sRenewal"].eq(TURNOVER).mean() * PCT,
    "sConcession == (sRentEffective < sRent) (% des baux)":
        (leases["sConcession"] == (leases["sRentEffective"] < leases["sRent"])).mean() * PCT,
}).round(1)
print(lease_column_checks.to_string(), end="\n\n")
print("Étiquette sLeaseTerm x durée réelle sTermMonths :")
pd.crosstab(leases["sLeaseTerm"], leases["sTermMonths"].round())

**Lecture :**

| Colonne | Constat | Ce qu'elle est réellement |
|---|---|---|
| `sAvailable` | égale à `sLeaseFrom` pour 100 % des baux | une copie de la date de début |
| `sTermSeq` | autant de `sTermSeq = 1` que d'unités, tous des relocations | le **rang du bail dans l'historique de l'unité**, pas du locataire |
| `sConcession` | vaut 1 exactement quand `sRentEffective < sRent` | un indicateur **dérivé** des deux loyers |
| `sTermMonths` | surtout 12 mois, parfois 6, 18 ou 24 | une durée calculée à partir des dates, fiable |
| `sLeaseTerm` | « 6 Months » ou « Short Term » contiennent des baux de 12 mois | une étiquette commerciale **non fiable** |

**Conséquences :** le type de bail vient de `sRenewal`, jamais de `sTermSeq`; `sConcession` n'est jamais utilisé à côté du
loyer effectif (même information deux fois); les hausses sont **annualisées** avec les dates réelles; `sAvailable` et
`sLeaseTerm` sont ignorés.

### 2c. Le piège `PromoPay`

La table des concessions décrit `PromoPay` comme un montant **mensuel**. On rattache chaque concession au bail de la même unité
dont la période contient son début, puis on exprime son montant en mois de loyer.

In [ ]:
def attach_concessions_to_leases(concession_df, lease_df):
    """Rattache chaque concession au bail de la même unité dont la période contient sDateFrom."""
    lease_columns = UNIT_KEY + ["sLeaseFrom", "sLeaseTo", "sRent"]
    candidates = (concession_df.reset_index(names="concession_id")
                  .merge(lease_df[lease_columns].reset_index(names="lease_id"), on=UNIT_KEY))
    starts_within_lease = candidates["sDateFrom"].between(candidates["sLeaseFrom"], candidates["sLeaseTo"])
    return candidates[starts_within_lease]


matched_concessions = attach_concessions_to_leases(concessions, leases)
concessions_by_code = (concessions.groupby("sChargeCode")
                       .agg(concessions=("sAmount", "size"),
                            median_amount=("sAmount", "median"),
                            median_months=("sMonths", "median"),
                            share_single_month=("sMonths", lambda months: (months == 1).mean()))
                       .sort_values("concessions", ascending=False))
print(f"concessions rattachées à un bail : {len(matched_concessions):,} / {len(concessions):,}\n")
print(concessions_by_code.round(2), end="\n\n")

promo = matched_concessions[matched_concessions["sChargeCode"] == PROMO_CHARGE_CODE]
promo_in_months_of_rent = -promo["sAmount"] / promo["sRent"]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(promo_in_months_of_rent, bins=40)
ax.set_xlabel("montant PromoPay / loyer mensuel du bail (en mois de loyer)")
ax.set_ylabel("concessions")
ax.set_title("PromoPay : un montant unique proche d'un mois de loyer, pas un rabais mensuel")
plt.tight_layout()
plt.show()

**Lecture :** `PromoPay` est inscrit sur **un seul mois** dans 88 % des cas, pour environ **un mois de loyer**. S'il était mensuel,
les résidents habiteraient presque gratuitement : la description est fausse. C'est un **mois gratuit unique à la signature**.
Les autres codes valent environ 130 à 150 $ par mois sur 3 à 12 mois; `freepark`, `freelock` et `freeappl` portent sur des
**accessoires**, pas sur le loyer.

**Conséquence :** `PromoPay` est renommé `one_time_signing_discount`; son effet doit être **étalé sur la durée du bail**
(un mois gratuit sur 12 mois ≈ 8 % de rabais). On vérifie que `sRentEffective` fait exactement cela.

### 2d. Validation de `sRentEffective`

Hypothèse : loyer effectif = loyer contractuel + (somme des concessions du bail, `sAmount × sMonths`) / durée du bail.

In [ ]:
concession_totals = (matched_concessions
                     .assign(total_amount=matched_concessions["sAmount"] * matched_concessions["sMonths"])
                     .groupby("lease_id")
                     .agg(concession_total=("total_amount", "sum"), concession_count=("total_amount", "size")))
reconstruction = leases.join(concession_totals).fillna({"concession_total": 0, "concession_count": 0})
observed_gap = reconstruction["sRentEffective"] - reconstruction["sRent"]
predicted_gap = reconstruction["concession_total"] / reconstruction["sTermMonths"]
reconstruction["is_reconstructed"] = (observed_gap - predicted_gap).abs() < RENT_MATCH_TOLERANCE

print(f"sRentEffective reconstruit au dollar près : {reconstruction['is_reconstructed'].mean():.1%} des baux\n")
failures = reconstruction[~reconstruction["is_reconstructed"]]
pd.crosstab(observed_gap[failures.index] < 0, failures["concession_count"] > 0,
            rownames=["rabais observé"], colnames=["concessions dans la table"])

**Lecture :** la formule reconstruit `sRentEffective` **au dollar près pour environ 62 % des baux** : une concordance exacte sur
des milliers de baux n'est pas une coïncidence. `sRentEffective` est bien le loyer contractuel diminué des concessions étalées
sur le bail, `PromoPay` compris. Les échecs viennent de la **table des concessions** : rabais sans concession dans l'extrait
(table incomplète), concessions sans rabais (surtout des accessoires non déduits du loyer), rattachement imparfait.

**Conséquence :** on utilise `sRentEffective` tel que calculé par Yardi, plutôt que de le recalculer à partir d'une table incomplète.

### 2e. Renommage des colonnes

**Cette étape modifie les tables** : à partir d'ici, les colonnes portent le nom de ce qu'elles contiennent réellement (2b à 2d).
Les clés et les colonnes descriptives (`sPropCode`, `sUnitCode`, `sBuilding`, `sState`, `sBeds`...) gardent leur nom Yardi, déjà exact.

In [ ]:
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",            # contractuel - concessions étalées sur le bail (2d)
    "sConcession": "has_concession",               # = effective_rent < contract_rent (2b)
    "sRenewal": "is_renewal",                      # 1 = locataire en place, 0 = relocation
    "sTermSeq": "unit_lease_rank",                 # rang du bail dans l'historique de l'unité (2b)
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",         # copie exacte de sLeaseFrom (2b)
    "sLeaseTerm": "lease_term_label_unreliable",   # étiquette incohérente avec la durée réelle (2b)
}
LISTING_COLUMN_RENAMES = {"sRent": "proposed_next_rent"}     # loyer proposé au prochain bail (section 7i)
CONCESSION_COLUMN_RENAMES = {"sAmount": "amount_per_month", "sMonths": "months_applied",
                             "sDateFrom": "concession_start", "sDateTo": "concession_end"}
CHARGE_CODE_LABELS = {
    "PromoPay": "one_time_signing_discount",       # PAS mensuel : environ un mois de loyer, une fois (2c)
    "freerent": "free_rent_monthly",
    "freepark": "free_parking_accessory",
    "freelock": "free_locker_accessory",
    "freeappl": "free_appliances_accessory",
    "Freeothe": "other_uncategorized",
    "Indem": "tenant_indemnity",
    "Referenc": "referral_credit",
}

leases = leases.rename(columns=LEASE_COLUMN_RENAMES)
listings = listings.rename(columns=LISTING_COLUMN_RENAMES)
concessions = concessions.rename(columns=CONCESSION_COLUMN_RENAMES)
concessions["charge_type"] = concessions["sChargeCode"].map(CHARGE_CODE_LABELS)
leases[list(LEASE_COLUMN_RENAMES.values())].head(3)

### 2f. L'effet de composition (mix)

Pourquoi la médiane annuelle de tous les baux ne mesure pas la hausse : on regarde **qui** est loué chaque année.

In [ ]:
analysis_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)].copy()
analysis_window["rent_per_sqft"] = analysis_window["contract_rent"] / analysis_window["sSqft"]

mix_by_year = analysis_window.groupby("lease_start_year").agg(
    leases=("contract_rent", "size"),
    buildings=("sBuilding", "nunique"),
    median_rent=("contract_rent", "median"),
    median_sqft=("sSqft", "median"),
    pct_two_bed_plus=("sBeds", lambda beds: (beds >= TWO_BEDROOMS).mean() * PCT),
    median_rent_per_sqft=("rent_per_sqft", "median"))
mix_by_year["naive_median_growth_pct"] = mix_by_year["median_rent"].pct_change() * PCT
print(mix_by_year.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
share_by_building = pd.crosstab(analysis_window["lease_start_year"], analysis_window["sBuilding"], normalize="index") * PCT
share_by_building.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set(xlabel="année de début du bail", ylabel="% des baux de l'année",
            title="Trois immeubles haut de gamme entrent en 2023-2024")
axes[0].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
analysis_window.groupby("sBuilding")["rent_per_sqft"].median().sort_values().plot(kind="barh", ax=axes[1])
axes[1].set(xlabel="loyer contractuel médian au pi² ($)", ylabel="",
            title="Les nouveaux immeubles sont beaucoup plus chers au pi²")
plt.tight_layout()
plt.show()

**Lecture :** jusqu'en 2022, seuls les trois immeubles de Laval génèrent des baux. Le Carlyle et The Met entrent en 2023,
Westpark en 2024. Les unités louées deviennent **plus petites** et comptent moins de 2 chambres et plus, ce qui devrait faire
*baisser* la médiane, et pourtant elle bondit de **10,8 % en 2023** : les nouveaux immeubles sont beaucoup plus chers au pi².
La médiane mélange des produits différents.

**Conséquence :** la médiane annuelle mesure la **composition du portefeuille**, pas la hausse. On compare chaque unité à
elle-même (section 3). **Limite :** avant 2023, l'historique ne couvre que Laval; le backtest 2023 prédit donc Mont-Royal et
Ottawa avec un historique lavallois.

## 3. Mesurer la croissance à unité constante

Trois étapes : **apparier** chaque bail au bail précédent de la même unité (`UNIT_KEY`), **filtrer** les paires non
comparables, **annualiser** :

$$\text{croissance annualisée} = \left(\frac{\text{loyer}}{\text{loyer précédent}}\right)^{1/\text{écart en années}} - 1$$

La puissance (plutôt qu'une division) tient compte de la composition des hausses : 1 000 $ → 1 060 $ en 18 mois donne
3,96 %/an, et non 6 %. Pour chaque paire, on calcule aussi le **rabais du bail précédent** et la **hausse du rabais**
(section 4), et on range la paire dans un **segment** : renouvellement au Québec, renouvellement en Ontario, ou relocation (section 5).

### 3a. Apparier, filtrer, annualiser

In [ ]:
RENT_COLUMNS = ["contract_rent", "effective_rent"]
PREVIOUS_LEASE_COLUMNS = RENT_COLUMNS + ["lease_start"]


def pair_consecutive_leases(lease_df, unit_key=UNIT_KEY):
    """Associe chaque bail au bail précédent de la même unité (colonnes `prev_`); le premier bail d'une unité n'a pas de paire."""
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired


def is_comparable(paired):
    """Écart entre MIN_GAP_YEARS et MAX_GAP_YEARS, et quatre loyers positifs."""
    has_comparable_gap = paired["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    rent_columns = RENT_COLUMNS + [f"prev_{rent}" for rent in RENT_COLUMNS]
    return has_comparable_gap & paired[rent_columns].gt(0).all(axis=1)


def annualized_growth_pct(paired, rent_column):
    ratio = paired[rent_column] / paired[f"prev_{rent_column}"]
    return (ratio ** (1 / paired["gap_years"]) - 1) * PCT


def discount_pct(effective_rent, contract_rent):
    """Part du loyer contractuel qui n'est pas perçue (%)."""
    return (1 - effective_rent / contract_rent) * PCT


def build_comparable_pairs(lease_df):
    """Paires comparables avec croissances annualisées, rabais du bail précédent, hausse du rabais et segment."""
    paired = pair_consecutive_leases(lease_df)
    pairs = paired[is_comparable(paired)].copy()
    for rent in RENT_COLUMNS:
        pairs[f"{rent}_growth_pct"] = annualized_growth_pct(pairs, rent)
    pairs["prev_discount_pct"] = discount_pct(pairs["prev_effective_rent"], pairs["prev_contract_rent"])
    pairs["discount_change_pts"] = discount_pct(pairs["effective_rent"], pairs["contract_rent"]) - pairs["prev_discount_pct"]
    is_renewal, is_ontario = pairs["is_renewal"].eq(RENEWAL), pairs["sState"].eq(ONTARIO)
    pairs["segment"] = np.select([is_renewal & ~is_ontario, is_renewal & is_ontario],
                                 ["renewal_QC", "renewal_ON"], default="turnover")
    return pairs


all_pairs = pair_consecutive_leases(leases)
comparable_pairs = build_comparable_pairs(leases)
pairs = comparable_pairs[comparable_pairs["lease_start_year"] >= FIRST_ANALYSIS_YEAR]

all_pairs["gap_bucket"] = pd.cut(all_pairs["gap_years"], bins=GAP_BINS_YEARS)
all_pairs["raw_contract_growth_pct"] = (all_pairs["contract_rent"] / all_pairs["prev_contract_rent"] - 1) * PCT
print(pd.Series({
    "baux": len(leases),
    "paires (bail précédent trouvé)": len(all_pairs),
    "paires comparables (filtres)": len(comparable_pairs),
    f"paires comparables depuis {FIRST_ANALYSIS_YEAR}": len(pairs),
}).to_string(), end="\n\n")
all_pairs.groupby("gap_bucket", observed=True).agg(
    pairs=("gap_years", "size"), median_raw_contract_growth_pct=("raw_contract_growth_pct", "median")).round(2)

**Lecture :** la grande majorité des paires sont espacées d'**un an**, le cas d'un bail de 12 mois suivi du suivant.
La hausse **brute** est à peu près la même quel que soit l'écart (environ 4,5 %) : le loyer augmente **par bail**, pas au
prorata du temps. Annualiser ramène donc les paires de 18-24 mois à leur vrai rythme annuel, ce qui compte pour un budget annuel.
Le filtre retire les 62 paires de moins de 6 mois (baux de 6 mois, dont la hausse annualisée doublerait artificiellement);
aucune paire ne dépasse 2,5 ans, mais le seuil protège la méthode.

### 3b. Ce que ferait la mauvaise clé (`sSite + sUnitCode`)

On apparie **une seule fois** sur la clé interdite, pour montrer l'effet.

In [ ]:
wrong_key_pairs = pair_consecutive_leases(leases, WRONG_UNIT_KEY)
raw_growth = lambda paired: (paired["contract_rent"] / paired["prev_contract_rent"] - 1) * PCT
pd.DataFrame({
    "sPropCode + sUnitCode": raw_growth(all_pairs).describe(),
    "sSite + sUnitCode (interdit)": raw_growth(wrong_key_pairs).describe(),
}).round(2)

**Lecture :** la mauvaise clé crée des paires qui n'existent pas et compare un appartement à un **autre** appartement du même
site : on voit apparaître des « baisses » et des « hausses » absurdes, et un écart-type plusieurs fois plus grand. Avec la bonne
clé, toutes les variations restent dans une plage plausible.

### 3c. Résultats : les définitions candidates chiffrées

In [ ]:
def median_by_year(pair_df, column, mask=None):
    selected = pair_df if mask is None else pair_df[mask]
    return selected.groupby("lease_start_year")[column].median()


analysis_years = list(range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1))
candidate_definitions = pd.DataFrame({
    "naive_median_all_leases": mix_by_year["naive_median_growth_pct"],
    "same_unit_contract": median_by_year(pairs, "contract_rent_growth_pct"),
    "same_unit_effective": median_by_year(pairs, "effective_rent_growth_pct"),
    "renewal_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(RENEWAL)),
    "turnover_effective": median_by_year(pairs, "effective_rent_growth_pct", pairs["is_renewal"].eq(TURNOVER)),
    "pairs": pairs.groupby("lease_start_year").size(),
}).loc[analysis_years]
yearly_target = candidate_definitions["same_unit_effective"]   # la cible (section 1)
print(candidate_definitions.round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
for column, style, label in [("naive_median_all_leases", "o:", "médiane naïve de tous les baux"),
                             ("same_unit_contract", "o-", "même unité, loyer contractuel"),
                             ("same_unit_effective", "s-", "même unité, loyer effectif (définition retenue)")]:
    ax.plot(analysis_years, candidate_definitions[column], style, lw=2, label=label)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="année de début du nouveau bail", ylabel="croissance annuelle (%)",
       title="La médiane naïve mesure le mix; contractuel et effectif divergent depuis 2024")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** la hausse réelle d'un même appartement est bien plus modeste que la médiane naïve (2023 : +10,8 % naïf contre
environ 2 % à unité constante). Contractuel et effectif évoluent ensemble jusqu'en 2023, puis divergent (section 4).
La cible oscille entre 1,6 % et 4,6 % sur 2019-2025, avec un creux en 2023-2024.

### 3d. Sensibilité aux choix de méthode

On recalcule la croissance effective en changeant **un choix à la fois**.

In [ ]:
all_pairs["raw_effective_growth_pct"] = (all_pairs["effective_rent"] / all_pairs["prev_effective_rent"] - 1) * PCT
all_pairs["annualized_effective_growth_pct"] = annualized_growth_pct(all_pairs, "effective_rent")
within_max_gap = all_pairs["gap_years"] < MAX_GAP_YEARS
reference_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > MIN_GAP_YEARS)]
with_six_month_pairs = all_pairs[within_max_gap & (all_pairs["gap_years"] > SIX_MONTH_GAP_YEARS)]


def trimmed_mean(values):
    low, high = values.quantile([TRIM_SHARE, 1 - TRIM_SHARE])
    return values[values.between(low, high)].mean()


by_year = lambda pair_df: pair_df.groupby("lease_start_year")
method_sensitivity = pd.DataFrame({
    "référence (médiane, annualisée, écart > 0,5 an)": by_year(reference_pairs)["annualized_effective_growth_pct"].median(),
    "inclure les baux de 6 mois": by_year(with_six_month_pairs)["annualized_effective_growth_pct"].median(),
    "sans annualisation": by_year(reference_pairs)["raw_effective_growth_pct"].median(),
    "moyenne au lieu de médiane": by_year(reference_pairs)["annualized_effective_growth_pct"].mean(),
    "moyenne tronquée": by_year(reference_pairs)["annualized_effective_growth_pct"].apply(trimmed_mean),
}).loc[REQUIRED_BACKTEST_YEARS].T
method_sensitivity["écart max vs référence"] = method_sensitivity.sub(method_sensitivity.iloc[0]).abs().max(axis=1)
method_sensitivity.round(2)

**Lecture :** aucune variante ne s'écarte de plus d'environ un quart de point de la référence : **la mesure est robuste** aux
choix de filtre, d'annualisation et de statistique. La moyenne simple est un peu plus haute, tirée par quelques fortes hausses,
ce qui justifie la médiane.

## 4. Concessions : loyer contractuel ou loyer effectif?

Pour chaque paire, le mécanisme est exact :

$$\frac{\text{effectif}_t}{\text{effectif}_{t-1}} = \frac{\text{contractuel}_t}{\text{contractuel}_{t-1}} \times \frac{1 - \text{rabais}_t}{1 - \text{rabais}_{t-1}}$$

Un rabais **stable** ne crée aucun écart entre les deux croissances; seul un rabais **qui grossit** d'un bail à l'autre en crée un.
On mesure donc la fréquence des concessions, leur taille, et surtout leur **variation sur les mêmes appartements**.

In [ ]:
leases_in_window = leases[leases["lease_start_year"].between(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR)]
leases_in_window = leases_in_window.assign(
    discount_pct=discount_pct(leases_in_window["effective_rent"], leases_in_window["contract_rent"]))
concessions_by_year = pd.DataFrame({
    "pct_leases_with_concession": leases_in_window.groupby("lease_start_year")["has_concession"].mean() * PCT,
    "mean_discount_pct": leases_in_window.groupby("lease_start_year")["discount_pct"].mean(),
    "contract_growth": candidate_definitions["same_unit_contract"],
    "effective_growth": candidate_definitions["same_unit_effective"],
    "median_discount_change_pts": median_by_year(pairs, "discount_change_pts"),
    "pct_pairs_discount_up": pairs.groupby("lease_start_year")["discount_change_pts"].apply(lambda change: (change > 0).mean() * PCT),
})
concessions_by_year["growth_gap_pts"] = concessions_by_year["contract_growth"] - concessions_by_year["effective_growth"]
print(concessions_by_year.round(2), end="\n\n")

discount_change_by_type = (pairs[pairs["lease_start_year"] >= REQUIRED_BACKTEST_YEARS[0]]
                           .pivot_table(index="lease_start_year", columns="is_renewal", values="discount_change_pts", aggfunc="median")
                           .rename(columns={RENEWAL: "renouvellement", TURNOVER: "relocation"}))
print("Hausse médiane du rabais (points), par type de bail :")
print(discount_change_by_type.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
concessions_by_year[["contract_growth", "effective_growth"]].plot(ax=axes[0], marker="o")
axes[0].legend(["loyer contractuel", "loyer effectif"])
axes[0].set(xlabel="année du nouveau bail", ylabel="croissance médiane (%)", title="Mêmes appartements, deux loyers")
concessions_by_year[["growth_gap_pts", "median_discount_change_pts"]].plot(ax=axes[1], marker="o")
axes[1].legend(["écart contractuel - effectif (points)", "hausse médiane du rabais (points)"])
axes[1].set(xlabel="année du nouveau bail", ylabel="points", title="L'écart suit la hausse du rabais d'un bail à l'autre")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture : pourquoi les deux croissances divergent à partir de 2024.**

1. **Les concessions deviennent à la fois plus fréquentes et plus généreuses** : la part des baux avec concession passe de 44-57 %
   (2019-2023) à 73 % en 2024 puis 85 % en 2025, et le rabais moyen de 2-4 % à 9,1 % du loyer contractuel.
2. **Sur les mêmes appartements, le rabais grossit d'un bail à l'autre** : hausse médiane nulle en 2023, puis +1,2 point en 2024
   et +3,0 points en 2025 (le rabais augmente pour 86 % des paires de 2025). C'est presque exactement l'écart entre les deux
   croissances : 1,6 point en 2024 et 3,5 points en 2025 (courbe de droite).
3. **Renouvellements et relocations sont touchés de la même façon** : les concessions servent autant à **retenir** les locataires
   en place qu'à en attirer de nouveaux. Elles se comportent comme une **politique commerciale globale**, présente dans tous les
   immeubles, anciens compris.

En résumé : depuis 2024, Équinoxe continue d'augmenter les loyers inscrits aux baux (+7,0 % en 2025), mais en redonne une part
croissante sous forme de concessions; le loyer perçu ne monte que de 3,6 %.

**Conséquences :** (1) on prévoit le **loyer effectif** (section 1); (2) l'**évolution des concessions** est la principale source
d'incertitude pour 2026. Le rabais du bail précédent est un candidat naturel comme variable du modèle; on le teste en section 7f.

## 5. Renouvellements, relocations et règles provinciales

Le type de bail est celui du **nouveau** bail de chaque paire (`is_renewal`). Les règles diffèrent :

| Cas | Québec (5 immeubles) | Ontario (The Met) |
|---|---|---|
| Renouvellement | hausse encadrée par le **TAL** : taux indicatif annuel, qui ne s'impose que si le locataire conteste | **ligne directrice** provinciale, sauf pour les logements occupés pour la première fois après le 15 novembre 2018, **exemptés** du plafond |
| Relocation | loyer de marché (avec la clause G du bail) | loyer de marché, sans plafond |

**On n'applique jamais le plafond d'une province à l'autre**, et on n'écrête aucune hausse : on mesure ce qui a été signé.

In [ ]:
SEGMENT_LABELS = {"renewal_QC": "renouvellement Québec", "renewal_ON": "renouvellement Ontario", "turnover": "relocation"}
recent_pairs = pairs[pairs["lease_start_year"] >= COMPARISON_YEARS[0]]
by_segment = recent_pairs.pivot_table(index="lease_start_year", columns="segment", aggfunc="median",
                                      values=["contract_rent_growth_pct", "effective_rent_growth_pct"])
segment_counts = recent_pairs.pivot_table(index="lease_start_year", columns="segment", values="effective_rent_growth_pct", aggfunc="size")
print("Paires par segment :")
print(segment_counts.fillna(0).astype(int), end="\n\n")
print("Croissance médiane (%) :")
print(by_segment.round(2))

### 5a. Renouvellements québécois et taux du TAL

Si les renouvellements suivaient le TAL, la « prime » (hausse contractuelle observée - taux du TAL) serait stable.

In [ ]:
regulatory_rates = pd.read_csv(REGULATORY_RATES_PATH).set_index("year")
renewal_vs_regulation = pd.DataFrame({
    "renewal_QC_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_QC")],
    "tal_rate": regulatory_rates["tal_unheated_pct"],
    "renewal_ON_contract_growth": by_segment[("contract_rent_growth_pct", "renewal_ON")],
    "ontario_guideline": regulatory_rates["ontario_guideline_pct"],
}).loc[COMPARISON_YEARS]
renewal_vs_regulation["premium_over_tal_pts"] = (renewal_vs_regulation["renewal_QC_contract_growth"]
                                                 - renewal_vs_regulation["tal_rate"])
renewal_vs_regulation["premium_over_guideline_pts"] = (renewal_vs_regulation["renewal_ON_contract_growth"]
                                                       - renewal_vs_regulation["ontario_guideline"])
renewal_vs_regulation.round(2)

**Lecture (Québec) :** la prime est **instable** : environ +4 points en 2021-2022 (TAL très bas, Équinoxe augmente bien au-dessus),
puis proche de 0 depuis 2023. Depuis que le TAL est élevé, les renouvellements québécois le suivent à peu près; avant, ils l'ignoraient.
Le taux du TAL n'est qu'indicatif : on ne l'impose donc pas comme plafond; on le teste comme variable du modèle (7f).

### 5b. The Met : la ligne directrice ontarienne s'applique-t-elle?

In [ ]:
the_met_leases = leases[leases["sState"] == ONTARIO]
the_met_renewals = pairs[pairs["segment"] == "renewal_ON"]
pd.Series({
    "premier bail observé à The Met": f"{the_met_leases['lease_start'].min():%Y-%m-%d}",
    "date d'exemption ontarienne (première occupation après)": f"{ONTARIO_EXEMPTION_DATE:%Y-%m-%d}",
    "renouvellements appariés à The Met": len(the_met_renewals),
    "part au-dessus de la ligne directrice de l'année (%)": round(
        (the_met_renewals["contract_rent_growth_pct"]
         > the_met_renewals["lease_start_year"].map(regulatory_rates["ontario_guideline_pct"])).mean() * PCT, 1),
}).to_frame("valeur")

**Lecture (Ontario) :** The Met génère ses premiers baux en 2023, bien après le 15 novembre 2018 : il est **vraisemblablement
exempté** de la ligne directrice. Les données vont dans ce sens : **81 % de ses renouvellements dépassent la ligne directrice de
l'année** (2,5 %), et la hausse contractuelle médiane des renouvellements atteint 6,3 % en 2025, ce qui serait impossible pour un
logement assujetti. Le statut exact reste à confirmer avec un document (date de première occupation).

**Conséquences pour la prévision :**
- les renouvellements et les relocations sont **séparés** dans le modèle (variable `lease_type`), et chaque immeuble garde sa
  propre part de renouvellements;
- le taux réglementaire est **propre à chaque province** : TAL pour les immeubles québécois, ligne directrice pour The Met. On ne
  l'impose **jamais comme plafond** (The Met est probablement exempté, le TAL n'est qu'indicatif); on le teste comme variable en 7f;
- The Met n'a des paires qu'à partir de 2024 : sa prévision repose en partie sur ce que le modèle apprend du Québec (section 8).

Écart entre types de baux : en 2022-2023 les renouvellements montent plus vite, en 2024-2025 ce sont les relocations
(loyers de marché en hausse), ce qui justifie de ne pas les fusionner sans le dire au modèle.

## 6. Données publiques : intégrer et réconcilier

| Source | Série | Marché | Années | Usage |
|---|---|---|---|---|
| Statistique Canada (18-10-0004) | IPC, ensemble (indice 2002 = 100) | Canada | 2021-2025 | **variable du modèle** (`general_cpi`) |
| SCHL, Enquête sur les logements locatifs (fichiers Excel 2021-2025) | taux d'inoccupation | RMR Montréal, Ottawa (partie Ontario) | 2021-2025 | **variable du modèle** (`vacancy_rate`), concessions (6c) |
| Statistique Canada (14-10-0460) | taux de chômage | RMR Montréal; Toronto en substitut pour Ottawa | 2021-2025 | **variable du modèle** (`unemployment_rate`) |
| SCHL (mêmes fichiers) | variation du loyer des 2 chambres, **échantillon fixe** | RMR Montréal, Ottawa (partie Ontario) | 2020-2025 | réconciliation (6b) |
| Statistique Canada (18-10-0004) | IPC, composante logement | Canada | 2021-2025 | réconciliation (6b) |
| TAL, gouvernement de l'Ontario | taux annuel d'augmentation / ligne directrice | Québec / Ontario | 2019-2026 | règles (5), variable testée (7f) |

L'échantillon fixe de la SCHL compare **les mêmes logements** d'une année à l'autre : c'est l'équivalent marché de notre mesure
à unité constante, en loyer contractuel. Le fichier de l'IPC ne contient que le Canada; Ottawa est absent du fichier de chômage,
d'où Toronto en substitut. L'inoccupation est celle du **marché**, jamais celle d'Équinoxe (que l'extrait ne permet pas de calculer).

### 6a. Chargement des séries publiques

In [ ]:
CMHC_MARKET = {QUEBEC: "Montréal", ONTARIO: "Ottawa — partie Ontario"}
MARKET_ORDER = [CMHC_MARKET[QUEBEC], CMHC_MARKET[ONTARIO]]   # Montréal sert de référence dans le modèle
CPI_GEO = "Canada"                     # IPC.csv ne contient que le Canada
CPI_ALL_ITEMS, CPI_SHELTER = "Ensemble", "Logement"
LABOUR_GEO = {QUEBEC: "Montréal, Québec", ONTARIO: "Toronto, Ontario"}     # Ottawa absent du fichier : Toronto en substitut
CMHC_FIXED_SAMPLE_COLUMNS = {"previous": "P", "current": "R"}   # tableau 1.0 : variation oct. t-2 -> t-1 et t-1 -> t
CMHC_FIRST_FILE_YEAR, CMHC_LAST_FILE_YEAR = 2021, LAST_OBSERVED_YEAR
MONTHS_PER_YEAR = 12
PROVINCES = [QUEBEC, ONTARIO]


def cmhc_fixed_sample_growth():
    '''Variation du loyer des 2 chambres, échantillon fixe (SCHL, tableau 1.0), par marché et année d'enquête.'''
    records = []
    for file_year in range(CMHC_FIRST_FILE_YEAR, CMHC_LAST_FILE_YEAR + 1):
        sheets = read_workbook(RAW_PUBLIC_DIR / f"rmr-canada-{file_year}-fr.xlsx")
        table = next(name for name in sheets if name.replace("Tableau", "Table") == "Table 1.0")
        for _, cells in sheets[table]:
            if cells.get("A") in CMHC_MARKET_LABELS:
                market = CMHC_MARKET_LABELS[cells["A"]]
                records.append({"market": market, "year": file_year, "growth": number(cells.get(CMHC_FIXED_SAMPLE_COLUMNS["current"]))})
                records.append({"market": market, "year": file_year - 1, "growth": number(cells.get(CMHC_FIXED_SAMPLE_COLUMNS["previous"]))})
    return pd.DataFrame(records).drop_duplicates(["market", "year"]).set_index(["market", "year"])["growth"]


def statcan_series(path, filters):
    '''Série StatCan filtrée, indexée par (géographie, période).'''
    table = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    for column, value in filters.items():
        table = table[table[column] == value]
    return table.set_index(["GÉO", "PÉRIODE DE RÉFÉRENCE"])["VALEUR"].astype(float)


def annual_mean(monthly_series, geo):
    '''Moyenne annuelle d'une série mensuelle (années complètes seulement).'''
    values = monthly_series.xs(geo)
    by_year = values.groupby(pd.PeriodIndex(values.index, freq="M").year)
    return by_year.mean()[by_year.size() == MONTHS_PER_YEAR]


cmhc_survey, _ = extract_public_data(RAW_PUBLIC_DIR)
cmhc_survey = cmhc_survey.set_index(["market", "year"])
fixed_sample_growth = cmhc_fixed_sample_growth()
cpi_all_items = annual_mean(statcan_series(CPI_PATH, {"Produits et groupes de produits": CPI_ALL_ITEMS}), CPI_GEO)
cpi_shelter = annual_mean(statcan_series(CPI_PATH, {"Produits et groupes de produits": CPI_SHELTER}), CPI_GEO)
unemployment = statcan_series(LABOUR_PATH, {"Caractéristiques de la population active": "Taux de chômage"})

signal_rows = []
for province in PROVINCES:
    unemployment_by_year = annual_mean(unemployment, LABOUR_GEO[province])
    regulatory_column = "tal_unheated_pct" if province == QUEBEC else "ontario_guideline_pct"
    for year in range(FIRST_PUBLIC_YEAR, FORECAST_YEAR + 1):
        signal_rows.append({
            "province": province, "year": year,
            "general_cpi": cpi_all_items.get(year, np.nan),
            "vacancy_rate": cmhc_survey["vacancy_rate"].get((CMHC_MARKET[province], year), np.nan),
            "unemployment_rate": unemployment_by_year.get(year, np.nan),
            "regulatory_rate": regulatory_rates[regulatory_column].get(year, np.nan),
            "cmhc_fixed_sample_growth": fixed_sample_growth.get((CMHC_MARKET[province], year), np.nan),
            "cpi_shelter_growth": (cpi_shelter.get(year, np.nan) / cpi_shelter.get(year - 1, np.nan) - 1) * PCT,
        })
market_signals = pd.DataFrame(signal_rows).set_index(["province", "year"]).sort_index()
market_signals.round(2)

### 6b. Réconcilier la tendance interne avec le marché

On compare, par marché, la hausse **contractuelle** d'Équinoxe à unité constante avec l'échantillon fixe de la SCHL (le loyer
**inscrit**, sans concessions, sur les mêmes logements) et l'IPC logement du Canada (contexte d'inflation), puis la hausse
**effective** d'Équinoxe. Les années ne sont pas tout à fait alignées (SCHL : octobre à octobre; IPC : moyenne annuelle;
Équinoxe : année de début du bail) : on compare des **ordres de grandeur et des tendances**.

In [ ]:
reconciliation_rows = []
for province in PROVINCES:
    provincial_pairs = pairs[pairs["sState"] == province]
    for year in COMPARISON_YEARS:
        year_pairs = provincial_pairs[provincial_pairs["lease_start_year"] == year]
        reconciliation_rows.append({
            "province": province, "year": year, "pairs": len(year_pairs),
            "equinoxe_contract": year_pairs["contract_rent_growth_pct"].median(),
            "cmhc_fixed_sample": market_signals.loc[(province, year), "cmhc_fixed_sample_growth"],
            "cpi_shelter_canada": market_signals.loc[(province, year), "cpi_shelter_growth"],
            "regulatory_rate": market_signals.loc[(province, year), "regulatory_rate"],
            "equinoxe_effective": year_pairs["effective_rent_growth_pct"].median(),
        })
reconciliation = pd.DataFrame(reconciliation_rows).set_index(["province", "year"])
print(reconciliation.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, province in zip(axes, PROVINCES):
    table = reconciliation.xs(province)
    ax.plot(table.index, table["equinoxe_contract"], "o-", lw=2, label="Équinoxe, contractuel (même unité)")
    ax.plot(table.index, table["equinoxe_effective"], "s-", lw=2, label="Équinoxe, effectif (même unité)")
    ax.plot(table.index, table["cmhc_fixed_sample"], "^--", label="SCHL, échantillon fixe 2 ch.")
    ax.plot(table.index, table["cpi_shelter_canada"], "d--", label="IPC logement, Canada")
    ax.step(table.index, table["regulatory_rate"], where="mid", color="grey", label="TAL / ligne directrice")
    ax.set(title=CMHC_MARKET[province], xlabel="année", ylabel="croissance annuelle (%)", xticks=COMPARISON_YEARS)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lecture :**

- **Québec, loyer contractuel : Équinoxe suit le marché, avec un décalage.** En 2022 et en 2025, la hausse contractuelle
  d'Équinoxe est au niveau de l'échantillon fixe de la SCHL (5,4 % contre 5,4 % en 2022; 7,1 % contre 7,2 % en 2025). En 2023-2024,
  le marché accélère (7,9 % et 6,3 %) mais Équinoxe ralentit (2,3 % et 3,4 %), puis rattrape en 2025.
- **Le creux de 2023 est propre à Équinoxe** : le marché accélérait. Aucune série publique ne pouvait l'annoncer, ce qui
  explique l'erreur du backtest 2023 (7e).
- **Loyer effectif : l'écart avec le marché, ce sont les concessions.** En 2024-2025, la hausse contractuelle rejoint le marché,
  mais Équinoxe en redonne une partie : la hausse effective (1,8 % puis 3,6 %) reste nettement sous le marché.
- **Ontario :** The Met dépasse le marché d'Ottawa en contractuel en 2025 (6,9 % contre 3,4 % pour la SCHL), cohérent avec un
  immeuble neuf exempté du plafond; en effectif (3,4 %), il rejoint le marché.
- **IPC logement (Canada) :** l'inflation du logement ralentit, de 6,9 % en 2022 à 3,1 % en 2025. C'est un contexte national et
  plus large que le loyer (il inclut l'intérêt hypothécaire et l'énergie) : il ne se compare pas directement à nos hausses.

**Conséquence :** la tendance interne et le marché sont **réconciliés** : l'écart vient de deux mécanismes identifiés, le décalage
de 2023-2024 et les concessions. Pour 2026, la question centrale est : « les concessions vont-elles continuer de grossir? ».

### 6c. Les concessions répondent-elles au marché?

Hypothèse : les concessions suivent le **taux d'inoccupation du marché**. On compare la hausse médiane du rabais (Québec) et la
variation de l'inoccupation à Montréal.

In [ ]:
quebec_pairs = pairs[pairs["sState"] == QUEBEC]
vacancy_montreal = market_signals.xs(QUEBEC)["vacancy_rate"]
discount_vs_vacancy = pd.DataFrame({
    "median_discount_change_pts": median_by_year(quebec_pairs, "discount_change_pts"),
    "vacancy_rate": vacancy_montreal,
    "vacancy_change_pts": vacancy_montreal.diff(),
}).loc[analysis_years].dropna()   # inoccupation SCHL disponible depuis 2021, variation depuis 2022
vacancy_correlation = discount_vs_vacancy[["median_discount_change_pts", "vacancy_change_pts"]].corr().iloc[0, 1]
print(discount_vs_vacancy.round(2))
print(f"\nCorrélation hausse du rabais / variation de l'inoccupation (même année) : {vacancy_correlation:.2f} "
      f"(n = {len(discount_vs_vacancy)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(discount_vs_vacancy["vacancy_change_pts"], discount_vs_vacancy["median_discount_change_pts"])
for year, row in discount_vs_vacancy.iterrows():
    ax.annotate(str(year), (row["vacancy_change_pts"], row["median_discount_change_pts"]), textcoords="offset points", xytext=(5, 5))
ax.set(xlabel="variation du taux d'inoccupation, Montréal (points, SCHL)", ylabel="hausse médiane du rabais (points)",
       title="Inoccupation du marché et concessions d'Équinoxe")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** l'inoccupation à Montréal baisse jusqu'en 2023 (1,5 %, marché très serré), puis remonte (2,1 % en 2024,
2,9 % en 2025). Les rabais d'Équinoxe suivent : stables quand le marché se resserre, en forte hausse quand il se détend
(corrélation de 0,79). C'est cohérent avec l'hypothèse : **les concessions répondent au marché**. Prudence : 4 années seulement,
et la relation est contemporaine (l'inoccupation d'octobre *t* n'est connue qu'à la fin de *t*).

**Comment ces données entrent dans la prévision (section 7) :**
- **IPC, inoccupation et chômage** sont des variables du modèle, **décalées d'un an** : pour prévoir l'année *t*, on n'utilise
  que leur valeur de *t* - 1, seule connue au moment de prévoir;
- le **taux réglementaire** est testé comme variable (7f);
- l'**échantillon fixe de la SCHL** et l'**IPC logement** servent à vérifier la prévision 2026 contre le marché (7i).

## 7. Prévoir et valider

**Règle du backtest :** pour prévoir l'année *t*, la méthode n'utilise que ce qui était **connu au début de *t*** : les paires
dont le nouveau bail a débuté avant *t*, le logement, le bail précédent, et les séries publiques de *t* - 1. On compare ensuite
à la médiane réellement observée en *t*.

### 7a. Point de départ : extrapolations de tendance

Trois règles simples, appliquées à la série de la cible (3c) : reprendre la hausse de l'année précédente, la moyenne des trois
dernières années, ou prolonger une tendance linéaire.

In [ ]:
TREND_WINDOW_YEARS = 3


def trend_forecasts(target_series, year):
    '''Extrapolations de la cible pour `year`, à partir des seules années antérieures.'''
    history = target_series[target_series.index < year].dropna()
    slope, intercept = np.polyfit(history.index, history.to_numpy(), deg=1)
    return {
        "hausse de l'an dernier": history.iloc[-1],
        f"moyenne des {TREND_WINDOW_YEARS} dernières années": history.iloc[-TREND_WINDOW_YEARS:].mean(),
        "tendance linéaire": slope * year + intercept,
    }


baseline_forecasts = pd.DataFrame({year: trend_forecasts(yearly_target, year) for year in REQUIRED_BACKTEST_YEARS}).T
baseline_errors = baseline_forecasts.sub(yearly_target.loc[REQUIRED_BACKTEST_YEARS], axis=0)
baseline_summary = pd.DataFrame({
    "MAE annuelle 2023-2025": baseline_errors.abs().mean(),
    f"prévision {FORECAST_YEAR}": pd.Series(trend_forecasts(yearly_target, FORECAST_YEAR)),
})
baseline_summary.round(2)

**Lecture :** ces règles se trompent de 1,4 à 2,1 points par an sur 2023-2025. Elles ne savent rien des appartements loués ni du
marché : elles servent de **référence à battre**.

### 7b. Le modèle : régression linéaire au niveau des paires de baux

Plutôt que de prévoir une série de quelques points annuels, on suit la définition : le modèle prédit **la hausse de chaque bail**
à partir de ce qu'on sait avant ce bail, puis on prend la **médiane** des prévisions.

| Variable | Pourquoi | Connue avant le bail? |
|---|---|---|
| `prev_effective_rent` | loyer effectif du bail précédent : niveau de départ | oui |
| `gap_years` | durée écoulée depuis le bail précédent | oui (fin du bail précédent) |
| `bedrooms` | taille du logement | oui |
| `market` | Montréal ou Ottawa (règles et marché différents, section 5) | oui |
| `lease_type` | renouvellement ou relocation (section 5) | **non** : voir ci-dessous |
| `general_cpi_prev_year` | niveau des prix (IPC) de l'année précédente | oui |
| `vacancy_rate_prev_year` | inoccupation du marché de l'année précédente (SCHL) | oui |
| `unemployment_rate_prev_year` | chômage du marché de l'année précédente | oui |

**Pourquoi une régression linéaire :** chaque coefficient se lit directement (7d), et elle obtient la meilleure erreur sur 2025
parmi trois familles de modèles (7c). Les facteurs publics n'existent qu'à partir de 2021 : avec le décalage d'un an, le modèle
apprend sur les paires de **2022 à l'année précédente**.

**Le type du prochain bail n'est pas connu d'avance** : chaque bail est prédit comme renouvellement **et** comme relocation,
pondéré par la part historique de renouvellements de son immeuble. **Population 2026** : le prochain bail de chaque logement
débute le lendemain de la fin du dernier bail connu; on retient ceux qui débutent en 2026, avec les mêmes filtres d'écart.

### 7c. `estimate_2026()`, `backtest()` et choix de la famille de modèle

In [ ]:
RETAINED_SPEC = {"private": PRIVATE_FEATURES, "lagged_public": PUBLIC_FACTORS, "current_year_public": []}


def market_signal(df, column, years_before=0):
    '''Valeur d'une série publique pour la province de chaque bail, `years_before` ans avant l'année du bail.'''
    keys = pd.MultiIndex.from_arrays([df["sState"], df["lease_start_year"] - years_before])
    return market_signals[column].reindex(keys).to_numpy()


def numeric_feature_names(spec):
    return (spec["private"] + [f"{factor}_prev_year" for factor in spec["lagged_public"]] + spec["current_year_public"])


def model_features(df, spec):
    '''Variables du modèle pour des paires (ou des baux à prévoir) : rien qui dépende du nouveau loyer.'''
    features = pd.DataFrame({
        "prev_effective_rent": df["prev_effective_rent"],
        "gap_years": df["gap_years"],
        "bedrooms": df["sBeds"].astype(float),
        "prev_discount_pct": df["prev_discount_pct"],
        "market": df["sState"].map(CMHC_MARKET),
        "lease_type": df["is_renewal"].map(LEASE_TYPE_LABELS),
    }, index=df.index)
    for factor in spec["lagged_public"]:
        features[f"{factor}_prev_year"] = market_signal(df, factor, years_before=PUBLIC_LAG_YEARS)
    for factor in spec["current_year_public"]:
        features[factor] = market_signal(df, factor)
    return features[numeric_feature_names(spec) + CATEGORICAL_FEATURES]


def make_model(spec, estimator=None):
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_feature_names(spec)),
        ("category", OneHotEncoder(categories=[MARKET_ORDER, list(LEASE_TYPE_LABELS.values())], drop="first",
                                   sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessing", preprocessing), ("regression", LinearRegression() if estimator is None else estimator)])


def training_set(history, year, spec):
    '''Paires antérieures à `year` dont toutes les variables sont connues.'''
    candidates = history[history["lease_start_year"].between(FIRST_ANALYSIS_YEAR, year - 1)]
    features = model_features(candidates, spec)
    complete = features.notna().all(axis=1)
    return candidates[complete], features[complete]


def weighted_median(values, weights):
    order = np.argsort(values)
    sorted_values, cumulative_weights = values[order], np.cumsum(weights[order])
    return sorted_values[np.searchsorted(cumulative_weights, cumulative_weights[-1] / 2)]


def predict_both_lease_types(model, population, training_pairs, spec):
    '''Prédit chaque bail comme renouvellement et comme relocation, pondéré par la part de renouvellements de l'immeuble.'''
    renewal_share = (population["sBuilding"].map(training_pairs.groupby("sBuilding")["is_renewal"].mean())
                     .fillna(training_pairs["is_renewal"].mean()).to_numpy())
    scenarios = []
    for lease_type, weight in [(RENEWAL, renewal_share), (TURNOVER, 1 - renewal_share)]:
        scenario = population.assign(is_renewal=lease_type)
        scenario["predicted_growth"] = model.predict(model_features(scenario, spec))
        scenario["weight"] = weight
        scenarios.append(scenario)
    return pd.concat(scenarios)


def next_lease_population(lease_df, year):
    '''Logements dont le prochain bail débute en `year` : le lendemain de la fin du dernier bail connu.'''
    last_leases = lease_df.sort_values("lease_start").groupby(UNIT_KEY).tail(1)
    next_start = last_leases["lease_end"] + pd.Timedelta(days=1)
    population = last_leases.assign(
        lease_start_year=next_start.dt.year,
        gap_years=(next_start - last_leases["lease_start"]).dt.days / DAYS_PER_YEAR,
        prev_effective_rent=last_leases["effective_rent"],
        prev_contract_rent=last_leases["contract_rent"],
        prev_discount_pct=discount_pct(last_leases["effective_rent"], last_leases["contract_rent"]))
    has_comparable_gap = population["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    return population[(population["lease_start_year"] == year) & has_comparable_gap]


def forecast_year(lease_df, year, population, spec):
    '''Entraîne sur les paires antérieures à `year`, prédit `population`; renvoie (médiane pondérée, prédictions, modèle, paires d'entraînement).'''
    training_pairs, training_features = training_set(build_comparable_pairs(lease_df), year, spec)
    model = make_model(spec).fit(training_features, training_pairs["effective_rent_growth_pct"])
    predictions = predict_both_lease_types(model, population, training_pairs, spec)
    growth = weighted_median(predictions["predicted_growth"].to_numpy(), predictions["weight"].to_numpy())
    return growth, predictions, model, training_pairs


def estimate_2026(leases, asking=None, external=None, spec=RETAINED_SPEC):
    '''Hausse 2026 (%) du loyer effectif à unité constante, médiane de tous les baux qui débutent en 2026 (section 1).

    `asking` n'est pas utilisé par la régression retenue (gardé pour la signature du gabarit); les séries publiques
    viennent de `market_signals` (section 6a).
    '''
    growth, _, _, _ = forecast_year(leases, FORECAST_YEAR, next_lease_population(leases, FORECAST_YEAR), spec)
    return float(growth)


def backtest(leases, target_year, spec=RETAINED_SPEC):
    '''Applique la méthode comme à la fin de target_year - 1, puis compare à la hausse observée en target_year.

    La population est celle des baux qui ont réellement débuté en target_year; on n'utilise que leur bail précédent,
    le logement et les séries publiques de target_year - 1. L'erreur par bail utilise le type de bail réel.
    '''
    history = build_comparable_pairs(leases)
    population = history[history["lease_start_year"] == target_year]
    forecast, _, model, _ = forecast_year(leases, target_year, population, spec)
    actual = population["effective_rent_growth_pct"].median()
    lease_errors = model.predict(model_features(population, spec)) - population["effective_rent_growth_pct"]
    return {"forecast": float(forecast), "actual": float(actual), "error": float(forecast - actual),
            "lease_mae": float(lease_errors.abs().mean())}

**Choisir la famille de modèle.** On compare, sur les mêmes variables, la régression linéaire à deux modèles d'arbres et à une
référence naïve (la moyenne historique). Entraînement sur les paires avant 2025, test sur les paires de 2025; erreurs **par bail**,
en points de pourcentage (plus faible = mieux).

In [ ]:
model_selection_pairs, model_selection_features = training_set(build_comparable_pairs(leases), MODEL_SELECTION_YEAR + 1, RETAINED_SPEC)
is_test = model_selection_pairs["lease_start_year"] == MODEL_SELECTION_YEAR
train_x, test_x = model_selection_features[~is_test], model_selection_features[is_test]
train_y, test_y = model_selection_pairs.loc[~is_test, "effective_rent_growth_pct"], model_selection_pairs.loc[is_test, "effective_rent_growth_pct"]

candidate_models = {
    "régression linéaire": make_model(RETAINED_SPEC),
    "forêt aléatoire": make_model(RETAINED_SPEC, RandomForestRegressor(**RANDOM_FOREST_PARAMS)),
    "gradient boosting": make_model(RETAINED_SPEC, GradientBoostingRegressor(**GRADIENT_BOOSTING_PARAMS)),
}
selection_rows = {"moyenne historique (référence)": np.full(len(test_y), train_y.mean())}
for name, candidate in candidate_models.items():
    selection_rows[name] = candidate.fit(train_x, train_y).predict(test_x)
model_selection = pd.DataFrame({
    name: {"MAE par bail": mean_absolute_error(test_y, predicted), "RMSE par bail": np.sqrt(mean_squared_error(test_y, predicted))}
    for name, predicted in selection_rows.items()}).T.sort_values("MAE par bail")
print(f"{len(train_y):,} paires d'entraînement ({model_selection_pairs.loc[~is_test, 'lease_start_year'].min()}-"
      f"{MODEL_SELECTION_YEAR - 1}), {len(test_y):,} paires de test ({MODEL_SELECTION_YEAR})\n")
model_selection.round(2)

**Lecture :** la régression linéaire a la plus faible erreur, devant les modèles d'arbres, et bat de peu la moyenne historique.
**Le gain est faible** : prévoir la hausse d'**un** bail précis reste très incertain (environ 4 points d'erreur). La médiane du
portefeuille, elle, est beaucoup plus stable : c'est elle qu'on valide en 7e.

### 7d. Ce que le modèle a appris : coefficients

Coefficients du modèle final (entraîné jusqu'en 2025). **Coefficient original** : points de hausse pour +1 unité de la variable;
**standardisé** : points pour +1 écart-type (comparable d'une variable à l'autre). Pour les catégories, l'écart à la référence
(Montréal, relocation). Ce sont des associations, pas des effets causaux.

In [ ]:
def coefficient_table(fitted, spec):
    preprocessing, regression = fitted.named_steps["preprocessing"], fitted.named_steps["regression"]
    scaler = preprocessing.named_transformers_["numeric"]
    numeric_names = numeric_feature_names(spec)
    category_names = list(preprocessing.named_transformers_["category"].get_feature_names_out(CATEGORICAL_FEATURES))
    numeric_coefficients = regression.coef_[:len(numeric_names)]
    table = pd.DataFrame({
        "coefficient_original": list(numeric_coefficients / scaler.scale_) + list(regression.coef_[len(numeric_names):]),
        "coefficient_standardise": regression.coef_,
    }, index=numeric_names + category_names)
    return table


forecast_2026, predictions_2026, final_model, final_training_pairs = forecast_year(
    leases, FORECAST_YEAR, next_lease_population(leases, FORECAST_YEAR), RETAINED_SPEC)
coefficients = coefficient_table(final_model, RETAINED_SPEC)
print(f"Modèle final : {len(final_training_pairs):,} paires ({final_training_pairs['lease_start_year'].min()}-{LAST_OBSERVED_YEAR})\n")
coefficients.round(4)

**Lecture :** 

- **Variables privées : des effets plausibles.** Un loyer précédent plus élevé est associé à une hausse plus faible (environ
  -1,3 point par 1 000 $), un écart plus long entre deux baux à une hausse annualisée plus faible, un logement plus grand à une
  hausse un peu plus forte. Les **renouvellements** montent d'environ 1,3 point de moins que les relocations, comme en 2024-2025
  (section 5). Ottawa ne se distingue presque pas de Montréal une fois ces variables prises en compte.
- **Facteurs publics : à lire avec prudence.** Les trois coefficients sont positifs. Pour l'inoccupation, le signe va à
  l'encontre de la section 6c : plus d'inoccupation, c'est plus de concessions, donc *moins* de hausse effective. Avec seulement
  quatre années de facteurs publics (2022-2025 une fois décalés), ces coefficients servent surtout à **distinguer les années**
  entre elles (2022 : forte hausse après une année 2021 de forte inoccupation et de fort chômage). Ce sont des signaux de contexte,
  pas des effets causaux; leur poids dans la prévision 2026 est mesuré en 7f.

### 7e. Backtest 2023-2025

In [ ]:
backtest_results = pd.DataFrame({year: backtest(leases, year) for year in REQUIRED_BACKTEST_YEARS}).T
model_mae = backtest_results["error"].abs().mean()
print(backtest_results.round(2), end="\n\n")

backtest_comparison = pd.concat([
    baseline_summary[["MAE annuelle 2023-2025"]],
    pd.DataFrame({"MAE annuelle 2023-2025": [model_mae]}, index=["régression linéaire (modèle retenu)"]),
])
print(backtest_comparison.round(2))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(yearly_target.index, yearly_target, "k-o", lw=2.5, label="réel (cible)")
ax.plot(backtest_results.index, backtest_results["forecast"], "--s", lw=2, label="régression linéaire")
for column, style in zip(baseline_forecasts.columns, [":^", ":v", ":d"]):
    ax.plot(baseline_forecasts.index, baseline_forecasts[column], style, alpha=0.6, label=column)
ax.set(xlabel="année", ylabel="croissance du loyer effectif, même unité (%)",
       title="Backtest : prévisions faites avec l'information disponible l'année précédente")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** 

- **La régression bat les extrapolations de tendance** : MAE de 0,99 point sur 2023-2025, contre 1,36 pour la meilleure
  règle simple.
- **2024 et 2025 sont prévus à 0,1 point près. 2023 reste la grande erreur** (4,9 % prévu pour 2,0 % réel) : le modèle n'a alors
  qu'une seule année d'entraînement (2022), et le ralentissement de 2023 est propre à Équinoxe (le marché accélérait, 6b).
- L'erreur **par bail** reste d'environ 4,2 à 4,5 points : la hausse d'un bail précis est très incertaine, mais la médiane du
  portefeuille, qui est notre définition, est beaucoup plus stable.

### 7f. Sensibilité aux variables

On refait le backtest et la prévision 2026 en retirant ou en ajoutant **un groupe de variables à la fois**.

In [ ]:
VARIANT_SPECS = {
    "retenu (privé + IPC, inoccupation, chômage)": RETAINED_SPEC,
    "sans inoccupation": {**RETAINED_SPEC, "lagged_public": ["general_cpi", "unemployment_rate"]},
    "sans IPC": {**RETAINED_SPEC, "lagged_public": ["vacancy_rate", "unemployment_rate"]},
    "sans chômage": {**RETAINED_SPEC, "lagged_public": ["general_cpi", "vacancy_rate"]},
    "privé seulement": {**RETAINED_SPEC, "lagged_public": []},
    "+ rabais précédent": {**RETAINED_SPEC, "private": PRIVATE_FEATURES + ["prev_discount_pct"]},
    "+ taux réglementaire": {**RETAINED_SPEC, "current_year_public": ["regulatory_rate"]},
}
variant_rows = {}
for variant, spec in VARIANT_SPECS.items():
    results = pd.DataFrame({year: backtest(leases, year, spec) for year in REQUIRED_BACKTEST_YEARS}).T
    variant_rows[variant] = {
        "MAE annuelle 2023-2025": results["error"].abs().mean(),
        "MAE par bail": results["lease_mae"].mean(),
        f"prévision {FORECAST_YEAR}": estimate_2026(leases, spec=spec),
    }
variants = pd.DataFrame(variant_rows).T
variants.round(2)

**Lecture :** 

- **Le modèle retenu a la meilleure erreur annuelle** (0,99 point) : c'est pourquoi on le garde.
- **Mais sa prévision 2026 dépend beaucoup des facteurs publics.** Sans l'inoccupation, elle tombe à 3,9 %; sans l'IPC, à 4,0 %;
  sans le chômage, le backtest se dégrade fortement (MAE de 4,3 points) et la prévision monte à 6,5 %. Avec seulement quatre
  années de facteurs publics, chaque facteur pèse lourd dans la façon dont le modèle distingue les années.
- **Variables privées seules : 2,9 %**, mais un backtest moins bon (1,45). Le **taux réglementaire** n'améliore pas le backtest (1,35).
- **Le rabais du bail précédent** améliore l'erreur par bail (4,02) mais dégrade l'erreur annuelle (2,62) et fait monter 2026 à
  7,1 % : les logements de 2026 partent d'un rabais moyen de 9 %, au-delà de l'historique, et une régression linéaire extrapole en
  ligne droite. Écarté.

**Conclusion :** on retient le modèle qui valide le mieux, et on rapporte sa sensibilité. La variante sans inoccupation (3,9 %)
sert de scénario prudent, puisque c'est le facteur dont l'effet appris contredit la section 6c.

### 7g. Prévision 2026

Modèle final entraîné sur toutes les paires jusqu'en 2025, avec les facteurs publics de 2025.

**Fourchette et scénarios :**
- **fourchette** : la prévision ± l'erreur annuelle typique du backtest (MAE 2023-2025);
- **sensibilité au facteur d'inoccupation** : la prévision sans ce facteur (7f), dont l'effet appris est le plus fragile;
- **type de bail** : la prévision si tous les baux 2026 étaient des renouvellements, ou tous des relocations.

In [ ]:
population_2026 = next_lease_population(leases, FORECAST_YEAR)
assert np.isclose(forecast_2026, estimate_2026(leases, asking))


def median_for_lease_type(predictions, lease_type):
    selected = predictions[predictions["is_renewal"] == lease_type]
    return float(selected["predicted_growth"].median())


scenarios_2026 = pd.Series({
    "bas (prévision - MAE)": forecast_2026 - model_mae,
    "central (modèle retenu)": forecast_2026,
    "haut (prévision + MAE)": forecast_2026 + model_mae,
    "sans le facteur d'inoccupation": variants.loc["sans inoccupation", f"prévision {FORECAST_YEAR}"],
    "tous renouvellements": median_for_lease_type(predictions_2026, RENEWAL),
    "toutes relocations": median_for_lease_type(predictions_2026, TURNOVER),
})
print(f"{len(population_2026):,} logements dont le prochain bail débute en {FORECAST_YEAR}\n")
print(f"Hausse {FORECAST_YEAR} prévue (loyer effectif, même unité, médiane) : {forecast_2026:.2f} %\n")
print(scenarios_2026.round(2).to_string())

MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model, "spec": RETAINED_SPEC, "features": numeric_feature_names(RETAINED_SPEC) + CATEGORICAL_FEATURES,
             "public_factors_2025": market_signals.xs(LAST_OBSERVED_YEAR, level="year")[PUBLIC_FACTORS]}, MODEL_PATH)
print(f"\nModèle sauvegardé : {MODEL_PATH}")

### 7h. Par immeuble et par nombre de chambres (bonus)

Chaque bail apparaît deux fois dans les prédictions (renouvellement et relocation, pondérés) : on prend la médiane pondérée par groupe.

In [ ]:
def forecast_by(predictions, group_column):
    rows = {}
    for group, group_predictions in predictions.groupby(group_column):
        rows[group] = {
            f"baux_{FORECAST_YEAR}": group_predictions.index.nunique(),
            "hausse_effective_prevue_pct": weighted_median(group_predictions["predicted_growth"].to_numpy(),
                                                           group_predictions["weight"].to_numpy()),
        }
    return pd.DataFrame(rows).T


forecast_by_building = forecast_by(predictions_2026, "sBuilding").sort_values("hausse_effective_prevue_pct")
forecast_by_bedrooms = forecast_by(predictions_2026, "sBeds")
print("Par immeuble :")
print(forecast_by_building.round(2), end="\n\n")
print("Par nombre de chambres :")
print(forecast_by_bedrooms.round(2))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1.3, 1.2, 1]})
axes[0].plot(yearly_target.index, yearly_target, "k-o", lw=2, label="réel")
axes[0].errorbar([FORECAST_YEAR], [forecast_2026], yerr=[[model_mae], [model_mae]], fmt="s", color="tab:red", capsize=6,
                 label=f"prévision {FORECAST_YEAR} (± MAE)")
axes[0].plot([FORECAST_YEAR], [scenarios_2026["sans le facteur d'inoccupation"]], "x", color="tab:grey", ms=9,
             label="sans le facteur d'inoccupation")
axes[0].set(title="Hausse du loyer effectif, même unité", xlabel="année", ylabel="%")
axes[0].legend(fontsize=8)

axes[1].barh(forecast_by_building.index, forecast_by_building["hausse_effective_prevue_pct"], color="tab:blue")
axes[1].axvline(forecast_2026, color="tab:red", ls="--", label="portefeuille")
for position, (building, row) in enumerate(forecast_by_building.iterrows()):
    axes[1].annotate(f"{row['hausse_effective_prevue_pct']:.1f} % ({row[f'baux_{FORECAST_YEAR}']:.0f} baux)",
                     (0, position), xytext=(4, -3), textcoords="offset points", fontsize=8, color="white")
axes[1].set(title=f"Prévision {FORECAST_YEAR} par immeuble", xlabel="%")
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))

axes[2].bar(forecast_by_bedrooms.index.astype(int).astype(str), forecast_by_bedrooms["hausse_effective_prevue_pct"], color="tab:green")
axes[2].axhline(forecast_2026, color="tab:red", ls="--")
axes[2].set(title=f"Prévision {FORECAST_YEAR} par nombre de chambres", xlabel="chambres", ylabel="%")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture :** les écarts entre immeubles viennent surtout du **niveau de loyer** et de la **taille des logements** :
Lévesque, aux loyers plus bas, a la prévision la plus haute (5,9 %), The Met la plus basse (4,8 %). Les studios (4,3 %) montent
moins que les logements d'une à trois chambres (5,2 % à 5,5 %). Tous les immeubles québécois partagent le même contexte public,
donc ces écarts restent modestes.

### 7i. Vérification de la prévision 2026 contre le marché

Les signaux de marché portent sur le loyer **contractuel**; notre prévision porte sur le loyer **effectif**. L'écart entre les deux
dépend des concessions (section 4) : si elles cessent de grossir, la hausse effective rejoint la hausse contractuelle; si elles
grossissent au rythme de 2025, elle reste près de la hausse effective de 2025.

In [ ]:
last_lease_rent = leases.sort_values("lease_start").groupby(UNIT_KEY).tail(1).set_index(UNIT_KEY)["contract_rent"]
listing_rents = listings.set_index(UNIT_KEY)["proposed_next_rent"]
proposed_markup_pct = (listing_rents / last_lease_rent.reindex(listing_rents.index) - 1) * PCT

market_check_2026 = pd.Series({
    f"prévision {FORECAST_YEAR}, loyer effectif (modèle)": forecast_2026,
    f"hausse effective Équinoxe {LAST_OBSERVED_YEAR} (concessions en hausse)": yearly_target.loc[LAST_OBSERVED_YEAR],
    f"hausse contractuelle Équinoxe {LAST_OBSERVED_YEAR} (si les concessions cessaient de grossir)":
        candidate_definitions.loc[LAST_OBSERVED_YEAR, "same_unit_contract"],
    "loyer proposé (listings) vs dernier bail, médiane": proposed_markup_pct.median(),
    f"TAL {FORECAST_YEAR} (renouvellements Québec)": market_signals.loc[(QUEBEC, FORECAST_YEAR), "regulatory_rate"],
    f"SCHL échantillon fixe Montréal {LAST_OBSERVED_YEAR}": market_signals.loc[(QUEBEC, LAST_OBSERVED_YEAR), "cmhc_fixed_sample_growth"],
    f"SCHL échantillon fixe Ottawa {LAST_OBSERVED_YEAR}": market_signals.loc[(ONTARIO, LAST_OBSERVED_YEAR), "cmhc_fixed_sample_growth"],
    f"IPC logement Canada {LAST_OBSERVED_YEAR}": market_signals.loc[(QUEBEC, LAST_OBSERVED_YEAR), "cpi_shelter_growth"],
})
market_check_2026.round(2).to_frame("%")

**Lecture :** 

- La prévision (5,3 %) se situe **entre** la hausse effective de 2025 (3,6 %, quand les concessions grossissaient de 3 points) et la
  hausse contractuelle de 2025 (7,0 %, ce qu'on obtiendrait si les concessions cessaient de grossir). Elle suppose donc que les
  concessions grossissent **nettement moins** qu'en 2025.
- C'est une hypothèse **optimiste** au regard des autres signaux : l'inoccupation du marché remontait encore en 2025 (6c), et les
  signaux contractuels propres à Équinoxe pour le prochain bail sont plus bas (loyer proposé dans `listings` environ 2,5 % au-dessus
  du dernier bail, TAL 2026 de 3,1 % pour les renouvellements québécois).
- Le scénario prudent (3,9 %, sans le facteur d'inoccupation) est proche de la hausse effective de 2025 et de ces signaux.

## 8. Hypothèses et limites

**Hypothèses**
1. Les relations apprises sur 2022-2025 entre les facteurs publics de l'année précédente et la hausse des loyers restent valables
   en 2026.
2. La **part de renouvellements** de chaque immeuble reste celle de l'historique.
3. Le prochain bail de chaque logement débute le lendemain de la fin du bail en cours (pas de vacance prolongée).
4. The Met est **exempté** de la ligne directrice ontarienne (immeuble occupé après le 15 novembre 2018), à confirmer.

**Limites**
1. **Quatre années de facteurs publics** (2021-2025, décalés d'un an) : les coefficients publics distinguent surtout des années
   entre elles. Le signe appris pour l'inoccupation (plus d'inoccupation, plus de hausse) va à l'encontre de la section 6c; sans
   ce facteur, la prévision serait de 3,9 % (7f).
2. **Les chocs d'année ne sont pas prévisibles avec ces données** : 2023 reste la grande erreur du backtest, et le marché
   accélérait cette année-là (6b).
3. **Le rabais du bail précédent** n'est pas dans le modèle : une régression linéaire l'extrapole en ligne droite, alors que les
   logements de 2026 partent d'un rabais (9 % en moyenne) au-delà de l'historique (7f).
4. **The Met** n'a des paires qu'à partir de 2024 : sa prévision repose surtout sur ce que le modèle apprend du Québec.
5. L'extrait ne permet pas de calculer l'occupation, l'inoccupation ni le revenu total du portefeuille : **aucun de ces chiffres
   n'est tiré des données d'Équinoxe**; l'inoccupation utilisée est celle du marché (SCHL).

## Conclusion

> **Hausse 2026 de Collection Équinoxe : 5,3 %** (croissance annualisée du loyer effectif à unité constante, médiane de
> tous les baux 2026, six immeubles), **fourchette 4,3 % à 6,3 %**. Si l'on retire le facteur d'inoccupation, dont l'effet
> appris est le plus fragile, la prévision descend à 3,9 %.

## 9. Références

**Données publiques** (fichiers dans `raw_public_data/` et `public_data/`, 2021-2025)
- SCHL, *Enquête sur les logements locatifs*, tableaux de données du Rapport sur le marché locatif 2021-2025 (`rmr-canada-AAAA-fr.xlsx`) :
  inoccupation, variation du loyer à échantillon fixe. https://www.cmhc-schl.gc.ca
- Statistique Canada, tableau 18-10-0004-01, *Indice des prix à la consommation mensuel*, Canada. https://www150.statcan.gc.ca
- Statistique Canada, tableau 14-10-0460-01, caractéristiques de la population active par RMR.
- Tribunal administratif du logement, calcul annuel de l'augmentation de loyer. https://www.tal.gouv.qc.ca
- Gouvernement de l'Ontario, ligne directrice sur l'augmentation des loyers et exemption des logements occupés après le
  15 novembre 2018. https://www.ontario.ca/page/rent-increase-guideline

**Méthode et outils**
- JADCO, *Clés en main* (consignes) et `starter.ipynb`.
- scikit-learn : `LinearRegression`, `RandomForestRegressor`, `GradientBoostingRegressor`. https://scikit-learn.org
- pandas, NumPy, matplotlib, joblib.

**Outils d'IA**
- Claude Code (Anthropic) : aide à l'écriture du code et des textes, à l'extraction des séries publiques, aux tests de variables et à la
  mise en forme du notebook. Les choix de méthode, les vérifications et les conclusions ont été revus par l'équipe.